# DBSCAN Experiments (Thesis-aligned) - Enhanced Version

Notebook ini fokus pada alur: 1) impor dan konfigurasi, 2) **duplicate analysis & normalization**, 3) k-distance plot untuk menentukan eps (dari unique samples jika perlu), 4) eksperimen parameter (eps & min_samples) menggunakan silhouette (sample), 5) fit final DBSCAN dan simpan model, 6) evaluasi cluster.

## 🆕 Enhanced Features:

1. **Cosine Distance Support** - Normalize embeddings untuk similarity yang lebih baik (especially untuk BERT embeddings)
2. **Smart Duplicate Handling** - Deteksi otomatis duplikat, compute eps dari unique samples, tapi fit DBSCAN pada FULL dataset (frequency preserved)
3. **Realistic Parameters** - min_samples 5-30 (bukan 3-12), KNN_NEIGHBORS = 20 (bukan 4)
4. **Better eps Range** - 75th-99th percentile (avoid excessive noise)
5. **Comprehensive Analysis** - Duplicate stats, k-distance dari unique samples, meaningful eps estimation

## 🎯 Problem Solved:

**Before:** k-distance = 0 for 90% of data (duplicates) → eps estimation fails  
**After:** k-distance computed from unique samples → meaningful eps → DBSCAN on full data (duplicates preserved for frequency info)

**Key Insight:** Duplicates = frequency information! Don't remove them, just compute eps smartly.


In [ ]:
# Part 1 — Imports & configuration
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import normalize
import pandas as pd
import gc
from tqdm import tqdm
import time

# ============================================================================
# CONFIGURATION - Specify your embedding file(s) directly
# ============================================================================

# Option 1: Single file (BGL or Thunderbird)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
    # Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_thunderbird_embeddings.npy"),
]

# Option 2: Multiple files (Combined dataset)
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_thunderbird_embeddings.npy"),
# ]

# Option 3: PCA variants (smaller, faster - RECOMMENDED for DBSCAN!)
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/after_preprocessed_bgl_pca256_embeddings.npy"),
# ]

# Option 4: PCA128 for ultra-large datasets
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca128/after_preprocessed_thunderbird_pca128_embeddings.npy"),
# ]

RANDOM_STATE = 42
SAMPLE_FOR_METRICS = 50000  # DBSCAN: reduce if memory limited
SAMPLE_FOR_KDIST = 500000   # Sample for k-distance plot (large datasets)
KNN_NEIGHBORS = 20          # for k-distance plot (k = min_samples, realistic for log embeddings)

# NEW: Duplicate & Distance handling
USE_COSINE_DISTANCE = True  # Normalize embeddings for cosine-like distance (recommended for embeddings)
HANDLE_DUPLICATES = True    # Compute k-distance from unique samples if many duplicates
DUPLICATE_THRESHOLD = 0.5   # If >50% duplicates, use unique samples for k-distance

print("📁 Input files configured:")
for f in INPUT_FILES:
    if f.exists():
        size_gb = f.stat().st_size / (1024**3)
        print(f"  ✓ {f.name} ({size_gb:.2f} GB)")
    else:
        print(f"  ❌ NOT FOUND: {f}")

print(f"\n⚙️ Configuration:")
print(f"  KNN_NEIGHBORS (min_samples): {KNN_NEIGHBORS}")
print(f"  Cosine distance: {'Enabled' if USE_COSINE_DISTANCE else 'Disabled'}")
print(f"  Duplicate handling: {'Enabled' if HANDLE_DUPLICATES else 'Disabled'}")


## Test: File Detection & Size Analysis

Quick diagnostic untuk verify files dan estimate runtime.

## 🔍 Quick Validation Test

Run this cell to quickly verify that the duplicate handling and normalization fixes work correctly. This will show you:
1. Whether duplicates are detected
2. Whether k-distance is meaningful (not all zeros)
3. Whether the configuration will work for your data

Skip this if you want to proceed directly to the full analysis.


In [ ]:
# Quick validation: Load small sample and test duplicate detection
print("🧪 Running quick validation test...\n")

# Load small sample
test_sample_size = 10000
if INPUT_FILES[0].exists():
    print(f"Loading {test_sample_size:,} samples for validation...")
    test_arr = np.load(INPUT_FILES[0], mmap_mode='r')
    n_available = test_arr.shape[0]
    
    if n_available < test_sample_size:
        test_sample_size = n_available
    
    test_sample = test_arr[:test_sample_size].copy()
    
    print(f"✓ Loaded: {test_sample.shape}")
    
    # Test 1: Normalization
    if USE_COSINE_DISTANCE:
        test_sample_norm = normalize(test_sample, norm='l2')
        print(f"\n1️⃣ Normalization test:")
        print(f"   Before: L2 norms range [{np.linalg.norm(test_sample, axis=1).min():.4f}, {np.linalg.norm(test_sample, axis=1).max():.4f}]")
        print(f"   After:  L2 norms range [{np.linalg.norm(test_sample_norm, axis=1).min():.4f}, {np.linalg.norm(test_sample_norm, axis=1).max():.4f}]")
        print(f"   ✅ Normalization working (all ~1.0)")
        test_data = test_sample_norm
    else:
        test_data = test_sample
        print(f"\n1️⃣ Normalization: DISABLED")
    
    # Test 2: Duplicate detection
    unique_test = np.unique(test_data, axis=0)
    dup_ratio = 1 - (len(unique_test) / len(test_data))
    print(f"\n2️⃣ Duplicate detection test:")
    print(f"   Sample size: {len(test_data):,}")
    print(f"   Unique:      {len(unique_test):,}")
    print(f"   Duplicates:  {dup_ratio:.1%}")
    
    if dup_ratio > DUPLICATE_THRESHOLD:
        print(f"   ✅ High duplicates detected → Will use unique samples for k-distance")
    else:
        print(f"   ✅ Low duplicates → Standard k-distance will work")
    
    # Test 3: k-distance computation
    print(f"\n3️⃣ k-distance test (k={min(5, len(unique_test))}):")
    if len(unique_test) >= 5:
        nn_test = NearestNeighbors(n_neighbors=5, metric='euclidean')
        nn_test.fit(unique_test)
        dist_test, _ = nn_test.kneighbors(unique_test)
        k_dist_test = dist_test[:, -1]
        
        print(f"   Min:    {k_dist_test.min():.6f}")
        print(f"   Median: {np.median(k_dist_test):.6f}")
        print(f"   Max:    {k_dist_test.max():.6f}")
        
        if np.sum(k_dist_test > 0) > len(k_dist_test) * 0.5:
            print(f"   ✅ k-distances are meaningful (>50% non-zero)")
        else:
            print(f"   ⚠️ Many zero k-distances - but will be handled by unique sample extraction")
    else:
        print(f"   ⚠️ Sample too small for k-distance test")
    
    print(f"\n{'='*60}")
    print(f"✅ Validation complete - Configuration looks good!")
    print(f"   Proceed to Cell 4 to run full analysis.")
    print(f"{'='*60}")
    
else:
    print(f"❌ File not found: {INPUT_FILES[0]}")
    print(f"   Please check INPUT_FILES configuration in Cell 2")


## 📈 Understanding k Parameter in k-distance Plot

### **What is k?**

**k = min_samples** yang akan digunakan di DBSCAN. k-distance plot menunjukkan jarak ke tetangga ke-k dari setiap point.

### **Why k=20 is Good for Log Embeddings?**

**Relationship between k and data:**
- **k too small (3-5)**: Overly sensitive, detects noise as patterns
- **k=10-30 (SWEET SPOT)**: Balanced for high-dimensional embeddings ✅
- **k too large (50+)**: Miss small anomaly patterns, only robust clusters

**Rule of thumb:**
- Low-dim data (< 50 dims): k = 3-5
- High-dim embeddings (128-768 dims): k = 10-30
- Log(dataset_size): For 4M samples → log(4M) ≈ 15 → **k=20 is optimal**

### **What Information Can We Extract from k-distance Plot?**

**From the curve shape:**
1. **Flat region** (left) = Dense/frequent patterns = Normal behavior
2. **Elbow point** = Transition point = **Recommended eps value**
3. **Steep rise** (right) = Sparse/rare patterns = Anomaly candidates

**From statistics:**
```
50th percentile: 0.34 → 50% of points have ≥20 neighbors within radius 0.34
90th percentile: 0.79 → 90% of points have ≥20 neighbors within radius 0.79
95th percentile: 0.92 → Top 5% are outliers (distance > 0.92)
```

**Choosing eps from k-distance:**
- **eps = 90th percentile** → 10% noise (balanced)
- **eps = 95th percentile** → 5% noise (more inclusive)
- **eps = 75th percentile** → 25% noise (strict, good for anomaly detection)

### **How to Validate k=20 is Good?**

Run the experiment cell below to compare different k values (10, 20, 30, 50) and see how eps recommendations change.


In [ ]:
# Experiment: Compare different k values to understand optimal min_samples
print("🧪 k-distance Comparison Experiment\n")
print("Comparing k=5, 10, 20, 30, 50 to find optimal min_samples for your data")
print("="*70)

# Load sample for k-distance comparison
if INPUT_FILES[0].exists():
    # Use smaller sample for speed
    k_comparison_sample_size = 50000
    print(f"\n📁 Loading {k_comparison_sample_size:,} samples for k-comparison...")
    
    test_arr = np.load(INPUT_FILES[0], mmap_mode='r')
    n_available = min(k_comparison_sample_size, test_arr.shape[0])
    
    # Sample randomly
    rng = np.random.RandomState(RANDOM_STATE)
    sample_idx = rng.choice(test_arr.shape[0], n_available, replace=False)
    k_comp_sample = test_arr[sample_idx].copy()
    
    # Normalize if enabled
    if USE_COSINE_DISTANCE:
        k_comp_sample = normalize(k_comp_sample, norm='l2')
    
    # Remove duplicates if enabled
    if HANDLE_DUPLICATES:
        k_comp_sample = np.unique(k_comp_sample, axis=0)
        print(f"✓ Using {len(k_comp_sample):,} unique samples after deduplication")
    else:
        print(f"✓ Loaded {len(k_comp_sample):,} samples")
    
    # Test different k values
    k_values = [5, 10, 20, 30, 50]
    k_results = []
    
    print(f"\n🔄 Computing k-distances for different k values...")
    for k in k_values:
        if k >= len(k_comp_sample):
            print(f"   ⚠️ Skipping k={k} (sample too small)")
            continue
            
        nn = NearestNeighbors(n_neighbors=k, n_jobs=-1, metric='euclidean')
        nn.fit(k_comp_sample)
        distances, _ = nn.kneighbors(k_comp_sample)
        k_dist = distances[:, -1]
        
        # Filter zeros
        k_dist_nz = k_dist[k_dist > 0]
        
        if len(k_dist_nz) > 0:
            k_results.append({
                'k': k,
                'p50': np.percentile(k_dist_nz, 50),
                'p60': np.percentile(k_dist_nz, 60),
                'p70': np.percentile(k_dist_nz, 70),
                'p75': np.percentile(k_dist_nz, 75),
                'p80': np.percentile(k_dist_nz, 80),
                'p85': np.percentile(k_dist_nz, 85),
                'p90': np.percentile(k_dist_nz, 90),
                'p95': np.percentile(k_dist_nz, 95),
                'mean': np.mean(k_dist_nz),
                'std': np.std(k_dist_nz)
            })
    
    # Display results
    df_k = pd.DataFrame(k_results)
    print("\n" + "="*70)
    print("K-DISTANCE COMPARISON RESULTS")
    print("="*70)
    print(df_k.to_string(index=False))
    
    # Interpretation
    print("\n" + "="*70)
    print("💡 INTERPRETATION")
    print("="*70)
    
    print("\n📊 Recommended eps for each k (using 85th, 90th, 95th percentile):")
    for _, row in df_k.iterrows():
        k_val = int(row['k'])
        eps_85 = row['p85']
        eps_90 = row['p90']
        eps_95 = row['p95']
        print(f"   k={k_val:2d} → eps ≈ {eps_85:.4f} (85%) | {eps_90:.4f} (90%) | {eps_95:.4f} (95%)")
    
    print("\n🎯 RECOMMENDATION:")
    
    # Find optimal k (look for stability)
    if len(k_results) >= 3:
        # Check eps stability across k values
        eps_90_values = [r['p90'] for r in k_results]
        eps_std = np.std(eps_90_values)
        eps_mean = np.mean(eps_90_values)
        cv = eps_std / eps_mean if eps_mean > 0 else 0
        
        print(f"   Coefficient of variation in eps (90%): {cv:.2%}")
        
        if cv < 0.15:
            print(f"   ✅ eps values are STABLE across k → Your data has consistent density")
            print(f"      → Use k=20-30 for balanced detection")
        elif cv < 0.30:
            print(f"   ⚠️ eps values show MODERATE variation")
            print(f"      → Use k=15-25 as middle ground")
        else:
            print(f"   ⚠️ eps values are HIGHLY VARIABLE")
            print(f"      → Your data has multiple density levels")
            print(f"      → Consider using smaller k (10-15) for sensitivity")
    
    # Visual plot
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Plot 1: eps recommendations with more percentiles
    ax1 = axes[0]
    x_pos = range(len(df_k))
    ax1.plot(df_k['k'], df_k['p70'], marker='d', label='70th percentile', linewidth=2, alpha=0.6)
    ax1.plot(df_k['k'], df_k['p80'], marker='s', label='80th percentile', linewidth=2, alpha=0.7)
    ax1.plot(df_k['k'], df_k['p85'], marker='p', label='85th percentile', linewidth=2, alpha=0.8)
    ax1.plot(df_k['k'], df_k['p90'], marker='o', label='90th percentile', linewidth=2.5)
    ax1.plot(df_k['k'], df_k['p95'], marker='^', label='95th percentile', linewidth=2)
    ax1.axvline(x=20, color='red', linestyle='--', alpha=0.5, label='k=20 (current)')
    ax1.set_xlabel('k (min_samples)')
    ax1.set_ylabel('Recommended eps')
    ax1.set_title('How eps Changes with Different k Values')
    ax1.legend(loc='best', fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Standard deviation
    ax2 = axes[1]
    ax2.bar(df_k['k'], df_k['std'], alpha=0.7, edgecolor='black')
    ax2.axvline(x=20, color='red', linestyle='--', alpha=0.5, label='k=20 (current)')
    ax2.set_xlabel('k (min_samples)')
    ax2.set_ylabel('Std Dev of k-distances')
    ax2.set_title('Distance Variability for Different k')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*70)
    print("✅ k-comparison complete!")
    print(f"   Current config (k={KNN_NEIGHBORS}) is {'GOOD' if 10 <= KNN_NEIGHBORS <= 30 else 'CONSIDER ADJUSTING'}")
    print("="*70)
    
else:
    print(f"❌ File not found: {INPUT_FILES[0]}")


## 🔍 Understanding Deduplication Impact

### **Why Deduplicate for k-distance Calculation?**

**Problem without deduplication:**
- If 90% of data are duplicates → 90% will have k-distance = 0
- eps recommendation = 0 → DBSCAN fails
- Can't distinguish between identical logs vs similar patterns

**Solution with deduplication:**
- Compute k-distance from UNIQUE patterns only
- Get meaningful eps = distance between DIFFERENT patterns
- Apply DBSCAN to FULL dataset → duplicates auto-cluster together

### **"Won't Data Become Too Sparse After Deduplication?"**

**Answer:** It depends on your data characteristics:

1. **If log patterns are limited** (good for clustering):
   - 1M logs → 1K unique patterns
   - Distance between patterns: 0.3 - 0.8 (reasonable)
   - ✅ eps = 0.6 works well

2. **If log patterns are very diverse** (challenging for clustering):
   - 1M logs → 500K unique patterns  
   - Distance between patterns: 0.8 - 2.5 (sparse!)
   - ⚠️ eps = 1.5 is correct - reflects real data diversity

**Key Insight:** If data becomes "far apart" after deduplication, that's the **REAL characteristic** of your logs, not an artifact!

### **Important: Final DBSCAN Uses FULL Dataset!**

```
Step 1: k-distance from unique samples → eps = 0.7
Step 2: DBSCAN on FULL data (with duplicates) → preserves frequency
Result: Duplicates auto-cluster (distance=0), large clusters = frequent = normal
```

Run the validation cell below to check the impact:


In [ ]:
# Validation: Compare k-distance WITH vs WITHOUT deduplication
print("🔬 Deduplication Impact Analysis\n")
print("Comparing k-distance distribution: Full data vs Unique samples")
print("="*70)

if INPUT_FILES[0].exists():
    # Load sample
    validation_sample_size = 100000
    print(f"\n📁 Loading {validation_sample_size:,} samples for validation...")
    
    test_arr = np.load(INPUT_FILES[0], mmap_mode='r')
    n_available = min(validation_sample_size, test_arr.shape[0])
    
    rng = np.random.RandomState(RANDOM_STATE)
    sample_idx = rng.choice(test_arr.shape[0], n_available, replace=False)
    val_sample = test_arr[sample_idx].copy()
    
    # Normalize if enabled
    if USE_COSINE_DISTANCE:
        val_sample = normalize(val_sample, norm='l2')
        print(f"✓ Normalized for cosine distance")
    
    print(f"✓ Loaded: {val_sample.shape}")
    
    # Extract unique
    val_unique = np.unique(val_sample, axis=0)
    n_total = len(val_sample)
    n_unique = len(val_unique)
    dup_ratio = 1 - (n_unique / n_total)
    
    print(f"\n📊 Sample Statistics:")
    print(f"   Total samples:     {n_total:,}")
    print(f"   Unique samples:    {n_unique:,}")
    print(f"   Duplicate ratio:   {dup_ratio:.1%}")
    print(f"   Data reduction:    {(1-n_unique/n_total)*100:.1f}x smaller")
    
    # Compute k-distance for both
    k_test = min(20, len(val_unique) - 1)
    
    print(f"\n🔄 Computing k-distance (k={k_test})...")
    
    # Scenario 1: WITH duplicates (problematic)
    print(f"\n1️⃣ K-distance WITH duplicates (full sample):")
    if n_total >= k_test:
        nn1 = NearestNeighbors(n_neighbors=k_test, n_jobs=-1, metric='euclidean')
        nn1.fit(val_sample)
        dist1, _ = nn1.kneighbors(val_sample)
        k_dist1 = dist1[:, -1]
        k_dist1_sorted = np.sort(k_dist1)
        
        # Stats
        zero_pct = (np.sum(k_dist1 == 0) / len(k_dist1)) * 100
        print(f"   Zero distances:     {zero_pct:.1f}%")
        print(f"   50th percentile:    {np.percentile(k_dist1, 50):.4f}")
        print(f"   75th percentile:    {np.percentile(k_dist1, 75):.4f}")
        print(f"   90th percentile:    {np.percentile(k_dist1, 90):.4f}")
        print(f"   95th percentile:    {np.percentile(k_dist1, 95):.4f}")
        
        if zero_pct > 50:
            print(f"   ❌ PROBLEM: >50% distances are 0 → eps unusable!")
    else:
        print(f"   ⚠️ Sample too small for k={k_test}")
        k_dist1_sorted = None
    
    # Scenario 2: WITHOUT duplicates (solution)
    print(f"\n2️⃣ K-distance WITHOUT duplicates (unique only):")
    if len(val_unique) >= k_test:
        nn2 = NearestNeighbors(n_neighbors=k_test, n_jobs=-1, metric='euclidean')
        nn2.fit(val_unique)
        dist2, _ = nn2.kneighbors(val_unique)
        k_dist2 = dist2[:, -1]
        k_dist2_sorted = np.sort(k_dist2)
        
        # Stats
        zero_pct2 = (np.sum(k_dist2 == 0) / len(k_dist2)) * 100
        print(f"   Zero distances:     {zero_pct2:.1f}%")
        print(f"   50th percentile:    {np.percentile(k_dist2, 50):.4f}")
        print(f"   75th percentile:    {np.percentile(k_dist2, 75):.4f}")
        print(f"   90th percentile:    {np.percentile(k_dist2, 90):.4f}")
        print(f"   95th percentile:    {np.percentile(k_dist2, 95):.4f}")
        
        if zero_pct2 < 10:
            print(f"   ✅ GOOD: <10% zeros → meaningful distance distribution!")
    else:
        print(f"   ⚠️ Sample too small for k={k_test}")
        k_dist2_sorted = None
    
    # Comparison
    if k_dist1_sorted is not None and k_dist2_sorted is not None:
        print(f"\n📈 COMPARISON:")
        
        eps1_90 = np.percentile(k_dist1, 90)
        eps2_90 = np.percentile(k_dist2, 90)
        
        print(f"   Recommended eps (90th percentile):")
        print(f"      With duplicates:    {eps1_90:.4f}")
        print(f"      Without duplicates: {eps2_90:.4f}")
        print(f"      Difference:         {abs(eps2_90 - eps1_90):.4f}")
        
        if eps1_90 < 0.01:
            print(f"\n   ❌ WITH duplicates: eps ≈ 0 → Cannot be used!")
            print(f"   ✅ WITHOUT duplicates: eps = {eps2_90:.4f} → Usable!")
        else:
            print(f"\n   ℹ️ Both methods give usable eps (low duplicate ratio)")
        
        # Visualize
        import matplotlib.pyplot as plt
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        
        # Plot 1: k-distance curves
        ax1 = axes[0]
        ax1.plot(k_dist1_sorted, label=f'With duplicates ({n_total:,} points)', linewidth=2, alpha=0.7)
        ax1.plot(k_dist2_sorted, label=f'Without duplicates ({n_unique:,} points)', linewidth=2, alpha=0.7)
        ax1.axhline(y=eps1_90, color='blue', linestyle=':', alpha=0.5, label=f'eps (with dup) = {eps1_90:.3f}')
        ax1.axhline(y=eps2_90, color='red', linestyle='--', alpha=0.5, label=f'eps (no dup) = {eps2_90:.3f}')
        ax1.set_xlabel('Points sorted by k-distance')
        ax1.set_ylabel(f'k-distance (k={k_test})')
        ax1.set_title('K-distance Comparison')
        ax1.legend(fontsize=8)
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Histogram comparison
        ax2 = axes[1]
        ax2.hist(k_dist1[k_dist1 > 0], bins=50, alpha=0.5, label='With duplicates', edgecolor='blue')
        ax2.hist(k_dist2[k_dist2 > 0], bins=50, alpha=0.5, label='Without duplicates', edgecolor='red')
        ax2.axvline(x=eps1_90, color='blue', linestyle=':', linewidth=2, label=f'eps (with) = {eps1_90:.3f}')
        ax2.axvline(x=eps2_90, color='red', linestyle='--', linewidth=2, label=f'eps (no dup) = {eps2_90:.3f}')
        ax2.set_xlabel('k-distance value')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distance Distribution (excluding zeros)')
        ax2.legend(fontsize=8)
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n💡 INTERPRETATION:")
        print(f"   • Left plot: Flat region at 0 in blue curve = duplicates")
        print(f"   • Red curve (unique): Shows true distance between DIFFERENT patterns")
        print(f"   • If red curve is 'far apart' → Your log patterns are diverse")
        print(f"   • This is REAL data characteristic, not deduplication artifact!")
        
        # Distance to closest DIFFERENT pattern
        print(f"\n🔍 Distance Analysis:")
        mean_dist_unique = np.mean(k_dist2[k_dist2 > 0])
        std_dist_unique = np.std(k_dist2[k_dist2 > 0])
        
        print(f"   Mean distance between unique patterns: {mean_dist_unique:.4f}")
        print(f"   Std dev:                                {std_dist_unique:.4f}")
        
        if mean_dist_unique > 1.0:
            print(f"   ⚠️ Patterns are relatively FAR APART → Diverse log behavior")
            print(f"      → Normal behavior less well-defined")
            print(f"      → Consider: Lower percentile for eps (75-80th) for tighter clusters")
        elif mean_dist_unique > 0.5:
            print(f"   ✅ Patterns at MODERATE distance → Good for clustering")
            print(f"      → Use 85-90th percentile for eps (balanced)")
        else:
            print(f"   ✅ Patterns are CLOSE → Well-defined log categories")
            print(f"      → Use 90-95th percentile for eps (inclusive)")
    
    print(f"\n{'='*70}")
    print(f"✅ Validation complete!")
    print(f"{'='*70}")
    
else:
    print(f"❌ File not found: {INPUT_FILES[0]}")


In [ ]:
# Diagnostic: Analyze configured files with auto-dimension detection
print("🔍 Analyzing configured files for DBSCAN...\n")

def detect_embedding_dim_diag(file_path: Path) -> int:
    """Auto-detect embedding dimension from filename pattern"""
    filename = file_path.name.lower()
    if 'pca256' in filename:
        return 256
    elif 'pca128' in filename:
        return 128
    else:
        return 768

def infer_num_rows_diag(path: Path, embedding_dim: int = None) -> int:
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim_diag(path)
    size = path.stat().st_size
    return size // (embedding_dim * np.dtype(np.float32).itemsize)

total_size_gb = 0
file_details = []

for f in INPUT_FILES:
    if not f.exists():
        print(f"❌ File not found: {f}")
        continue
        
    size_gb = f.stat().st_size / (1024**3)
    total_size_gb += size_gb
    
    # Auto-detect dimension from filename
    auto_dim = detect_embedding_dim_diag(f)
    
    try:
        test_arr = np.load(f, mmap_mode='r')
        file_type = "Standard .npy"
        shape = test_arr.shape
        actual_dim = test_arr.shape[1]
        del test_arr
        
        # Verify auto-detection matches
        if actual_dim != auto_dim:
            print(f"⚠️ Dimension mismatch for {f.name}:")
            print(f"   Auto-detected: {auto_dim}, Actual: {actual_dim}")
            print(f"   Using actual dimension from .npy header")
    except:
        file_type = "RAW memmap"
        n_rows = infer_num_rows_diag(f, embedding_dim=auto_dim)
        shape = (n_rows, auto_dim)
        actual_dim = auto_dim
    
    file_details.append({
        'name': f.name,
        'size_gb': size_gb,
        'type': file_type,
        'shape': shape,
        'dimension': actual_dim
    })

if len(file_details) == 0:
    print("⚠️ No valid files found! Check INPUT_FILES configuration.")
else:
    df = pd.DataFrame(file_details)
    print(df.to_string(index=False))
    
    print(f"\n{'='*60}")
    print(f"Total files: {len(file_details)}")
    print(f"Total size: {total_size_gb:.2f} GB")
    print(f"Total samples: {sum(d['shape'][0] for d in file_details):,}")
    
    # DBSCAN-specific warnings
    print(f"\n📊 DBSCAN RUNTIME ESTIMATE:")
    if total_size_gb < 5:
        print("✅ Small dataset (<5GB)")
        print("   → Expected time: 10-20 minutes")
        print("   → Memory: ~{:.1f} GB RAM".format(total_size_gb * 2))
    elif total_size_gb < 20:
        print("⚠️  Medium dataset (5-20GB)")
        print("   → Expected time: 30-60 minutes")
        print("   → Memory: ~{:.1f} GB RAM".format(total_size_gb * 2))
    elif total_size_gb < 100:
        print("🔥 Large dataset (20-100GB)")
        print("   → Expected time: 1-3 hours")
        print("   → Memory: ~{:.1f} GB RAM".format(total_size_gb * 2))
        print("   ⚠️ Consider using PCA variants!")
    else:
        print("❌ ULTRA LARGE dataset (>100GB)")
        print("   → Expected time: 4-8+ hours")
        print("   → Memory: ~{:.1f} GB RAM".format(total_size_gb * 2))
        print("   ⚠️ STRONGLY RECOMMEND: Use PCA128 variant!")
    
    print(f"\n💡 RECOMMENDATION:")
    if total_size_gb > 20:
        print("   Switch to PCA variant for faster processing:")
        print("   • PCA256: ~6x smaller, minimal quality loss")
        print("   • PCA128: ~12x smaller, acceptable quality loss")


In [ ]:
# Part 2 — Smart file loading with duplicate analysis and k-distance plot to estimate eps

def detect_embedding_dim(file_path: Path) -> int:
    """
    Auto-detect embedding dimension from filename pattern
    - *pca256* → 256 dims
    - *pca128* → 128 dims
    - default → 768 dims
    """
    filename = file_path.name.lower()
    if 'pca256' in filename:
        return 256
    elif 'pca128' in filename:
        return 128
    else:
        return 768

def infer_num_rows(path: Path, embedding_dim: int = None) -> int:
    """
    Infer number of rows for RAW memmap files
    Auto-detects dimension from filename if not provided
    """
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(path)
    size = path.stat().st_size
    return size // (embedding_dim * np.dtype(np.float32).itemsize)

def load_single_file_smart(file_path: Path, embedding_dim: int = None):
    """
    Smart loader: auto-detect .npy vs RAW memmap
    Returns (array, is_memmap, num_rows)
    
    Auto-detects dimension from filename if not provided:
    - *pca256* → 256 dims
    - *pca128* → 128 dims  
    - default → 768 dims
    """
    # Auto-detect dimension if not provided
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(file_path)
        print(f"   🔍 Auto-detected dimension: {embedding_dim} from filename")
    
    try:
        arr = np.load(file_path, mmap_mode='r')
        detected_dim = arr.shape[1]
        if detected_dim != embedding_dim:
            print(f"   ⚠️ Dimension mismatch! Expected {embedding_dim}, got {detected_dim} from .npy header")
            embedding_dim = detected_dim
        return arr, True, arr.shape[0]
    except Exception:
        num_rows = infer_num_rows(file_path, embedding_dim)
        arr = np.memmap(
            file_path, 
            dtype=np.float32, 
            mode='r', 
            shape=(num_rows, embedding_dim)
        )
        print(f"   ⚠️ Loaded as RAW memmap: {num_rows:,} rows × {embedding_dim} dims")
        return arr, True, num_rows

def load_embeddings_from_files(files, force_copy=False):
    """Load embeddings from list of file paths with auto-dimension detection"""
    if len(files) == 0:
        raise FileNotFoundError('No embedding files provided')
    
    for f in files:
        if not f.exists():
            raise FileNotFoundError(f'File not found: {f}')
    
    # Auto-detect dimension from first file
    first_arr, _, _ = load_single_file_smart(files[0])
    embedding_dim = first_arr.shape[1]
    print(f"Embedding dimension: {embedding_dim}")
    
    if len(files) == 1:
        print(f"Loading single file: {files[0].name}")
        return first_arr
    
    print(f"Loading {len(files)} files...")
    total_samples = 0
    file_info = []
    for f in files:
        arr, is_mmap, n_rows = load_single_file_smart(f)  # Auto-detect dimension
        file_info.append((f, arr, n_rows))
        total_samples += n_rows
        print(f"  - {f.name}: {n_rows:,} rows")
    
    total_size_gb = (total_samples * embedding_dim * 4) / (1024**3)
    print(f"\nTotal samples: {total_samples:,} ({total_size_gb:.2f} GB)")
    
    if total_size_gb < 100:
        print("Strategy: Memory-mapped stacking")
        arrays = [arr for _, arr, _ in file_info]
        return np.vstack(arrays)
    else:
        raise MemoryError(
            f"Dataset too large ({total_size_gb:.1f}GB). "
            "For DBSCAN, use single file or PCA variants (smaller size)"
        )

# Load embeddings
print("Loading embeddings...")
emb = load_embeddings_from_files(INPUT_FILES)
print(f'Loaded embeddings shape: {emb.shape}')

# NEW: Normalize for cosine distance
if USE_COSINE_DISTANCE:
    print('\n🔄 Normalizing embeddings for cosine-like distance...')
    emb = normalize(emb, norm='l2')
    print(f'✓ Embeddings normalized (using Euclidean on normalized = cosine distance)')

# NEW: Duplicate analysis
print('\n🔍 Analyzing duplicates...')
n_total = emb.shape[0]
if n_total > SAMPLE_FOR_KDIST:
    print(f'   Using sample of {SAMPLE_FOR_KDIST:,} for duplicate check')
    rng = np.random.RandomState(RANDOM_STATE)
    dup_check_idx = rng.choice(n_total, min(SAMPLE_FOR_KDIST, n_total), replace=False)
    emb_dup_check = emb[dup_check_idx]
else:
    emb_dup_check = emb

unique_embeddings = np.unique(emb_dup_check, axis=0)
n_unique = len(unique_embeddings)
n_sample = len(emb_dup_check)
duplicate_ratio = 1 - (n_unique / n_sample)

print(f'\n📊 Duplicate Analysis (sample):')
print(f'   Total samples checked: {n_sample:,}')
print(f'   Unique samples: {n_unique:,}')
print(f'   Duplicate ratio: {duplicate_ratio:.2%}')

# Decide whether to use unique samples for k-distance
use_unique_for_kdist = HANDLE_DUPLICATES and duplicate_ratio > DUPLICATE_THRESHOLD

if use_unique_for_kdist:
    print(f'\n⚠️ High duplicate ratio detected ({duplicate_ratio:.1%} > {DUPLICATE_THRESHOLD:.0%})')
    print(f'   → Computing k-distance from UNIQUE samples only')
    print(f'   → eps will be meaningful (not 0)')
    print(f'   → Final DBSCAN will run on FULL dataset (with duplicates)')
    
    # Get unique samples from full dataset (or large sample)
    if n_total > SAMPLE_FOR_KDIST * 2:
        print(f'   Sampling {SAMPLE_FOR_KDIST:,} from full dataset first...')
        sample_idx = rng.choice(n_total, SAMPLE_FOR_KDIST, replace=False)
        emb_sample = emb[sample_idx]
        unique_for_kdist = np.unique(emb_sample, axis=0)
    else:
        print(f'   Extracting unique from full dataset...')
        unique_for_kdist = np.unique(emb, axis=0)
    
    print(f'   Unique samples for k-distance: {len(unique_for_kdist):,}')
    emb_for_kdist = unique_for_kdist
else:
    print(f'\n✅ Low duplicate ratio ({duplicate_ratio:.1%})')
    print(f'   → Using standard sampling for k-distance')
    # For very large datasets, sample for k-distance plot
    if n_total > SAMPLE_FOR_KDIST:
        print(f'   Using sample of {SAMPLE_FOR_KDIST:,} for k-distance plot')
        rng = np.random.RandomState(RANDOM_STATE)
        kdist_idx = rng.choice(n_total, SAMPLE_FOR_KDIST, replace=False)
        emb_for_kdist = emb[kdist_idx]
    else:
        emb_for_kdist = emb

# Compute nearest-neighbors distances (k-distance)
print(f'\n🔄 Computing k-distance (k={KNN_NEIGHBORS})...')
print(f'   Sample size: {len(emb_for_kdist):,}')
nn = NearestNeighbors(n_neighbors=KNN_NEIGHBORS, n_jobs=-1, metric='euclidean')
nn.fit(emb_for_kdist)
distances, _ = nn.kneighbors(emb_for_kdist)
# distances[:, -1] is the distance to k-th neighbor
k_dist = np.sort(distances[:, -1])

# Filter out zeros if present
k_dist_nonzero = k_dist[k_dist > 0]
if len(k_dist_nonzero) < len(k_dist):
    pct_zero = (1 - len(k_dist_nonzero) / len(k_dist)) * 100
    print(f'   ⚠️ {pct_zero:.1f}% of k-distances are 0 (duplicates in unique sample)')
    print(f'   → Using non-zero distances only for statistics')
    k_dist_for_stats = k_dist_nonzero
else:
    k_dist_for_stats = k_dist

# Plot k-distance curve
plt.figure(figsize=(10,4))
plt.plot(k_dist)
plt.xlabel('Points sorted by k-distance')
plt.ylabel(f'k-distance (k={KNN_NEIGHBORS})')
title = 'k-distance plot — look for elbow to choose eps'
if use_unique_for_kdist:
    title += ' (computed from UNIQUE samples)'
plt.title(title)

if len(k_dist_for_stats) > 0:
    plt.axhline(y=np.percentile(k_dist_for_stats, 85), color='purple', linestyle=':', alpha=0.4, label='85th percentile')
    plt.axhline(y=np.percentile(k_dist_for_stats, 90), color='r', linestyle='--', alpha=0.5, label='90th percentile')
    plt.axhline(y=np.percentile(k_dist_for_stats, 95), color='orange', linestyle='--', alpha=0.5, label='95th percentile')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n📊 k-distance statistics:')
if len(k_dist_for_stats) > 0:
    # Display requested percentiles: 50, 60, 70, 80, 85, 90, 95
    percentiles = [50, 60, 70, 80, 85, 90, 95]
    for p in percentiles:
        val = np.percentile(k_dist_for_stats, p)
        print(f'   {p:2d}th percentile: {val:.4f}')
else:
    print('   ⚠️ All k-distances are 0!')
    print('   → Try: increase SAMPLE_FOR_KDIST, decrease KNN_NEIGHBORS, or enable normalization')

# Store k_dist_for_stats for next cell
k_dist_clean = k_dist_for_stats


In [ ]:
# Part 3 — Parameter search over eps and min_samples (uses a sample for silhouette)
# WARNING: DBSCAN can be slow for large datasets; we sample for metric computation

print('Preparing sample for parameter search...')
n = emb.shape[0]
if n > SAMPLE_FOR_METRICS:
    rng = np.random.RandomState(RANDOM_STATE)
    sample_idx = rng.choice(n, SAMPLE_FOR_METRICS, replace=False)
    emb_sample = emb[sample_idx]
    print(f'Using sample: {SAMPLE_FOR_METRICS:,} / {n:,} samples')
else:
    emb_sample = emb
    print(f'Using full dataset: {n:,} samples')

# Define parameter grid based on k-distance statistics
if len(k_dist_clean) > 0:
    eps_min = np.percentile(k_dist_clean, 75)  # Start higher to avoid too many noise points
    eps_max = np.percentile(k_dist_clean, 99)  # Extended to 99th percentile
    
    # If range is too small, expand it
    if eps_max - eps_min < 0.01:
        eps_min = np.percentile(k_dist_clean, 50)
        eps_max = np.percentile(k_dist_clean, 99)
    
    eps_values = np.linspace(eps_min, eps_max, 8)
else:
    print('⚠️ No valid k-distances, using default eps range')
    eps_values = np.linspace(0.1, 2.0, 8)

# Updated min_samples: more realistic values for log embeddings with duplicates
min_samples_values = [5, 10, 15, 20, 30]

print(f'\nParameter grid:')
print(f'  eps: {len(eps_values)} values from {eps_values.min():.4f} to {eps_values.max():.4f}')
print(f'  min_samples: {min_samples_values}')
print(f'  Total combinations: {len(eps_values) * len(min_samples_values)}')

results = []

total_combinations = len(eps_values) * len(min_samples_values)
print(f'\n🔄 Testing {total_combinations} parameter combinations...')

pbar = tqdm(total=total_combinations, desc='DBSCAN grid search', unit='config')
for eps_idx, eps in enumerate(eps_values, 1):
    for ms in min_samples_values:
        start_time = time.time()
        
        # n_jobs=-1 to use all CPU cores
        dbs = DBSCAN(eps=float(eps), min_samples=int(ms), n_jobs=-1, metric='euclidean')
        labels = dbs.fit_predict(emb_sample)
        
        # Compute number of clusters (exclude noise label -1)
        unique_labels = set(labels) - {-1}
        n_clusters = len(unique_labels)
        n_noise = np.sum(labels == -1)
        noise_pct = (n_noise / len(labels)) * 100
        
        sil = -1
        if n_clusters > 1:
            try:
                # n_jobs=-1 to maximize CPU usage
                sil = silhouette_score(emb_sample, labels, n_jobs=-1)
            except Exception:
                sil = -1
        
        results.append({
            'eps': float(eps),
            'min_samples': int(ms),
            'n_clusters': n_clusters,
            'noise_pct': float(noise_pct),
            'silhouette': float(sil)
        })
        
        elapsed = time.time() - start_time
        pbar.update(1)
        tqdm.write(f'  eps={eps:.4g}, min_samples={ms:2d} → clusters={n_clusters:2d}, noise={noise_pct:5.1f}%, sil={sil:6.4f} ({elapsed:.1f}s)')

pbar.close()

# Show results sorted by silhouette
df_res = pd.DataFrame(results)
print('\n' + '='*70)
print('TOP 10 CONFIGURATIONS (by silhouette score)')
print('='*70)
print(df_res.sort_values('silhouette', ascending=False).head(10).to_string(index=False))

# Also show configurations with reasonable noise levels
print('\n' + '='*70)
print('CONFIGURATIONS WITH LOW NOISE (<30%)')
print('='*70)
df_low_noise = df_res[df_res['noise_pct'] < 30].sort_values('silhouette', ascending=False)
if len(df_low_noise) > 0:
    print(df_low_noise.head(10).to_string(index=False))
else:
    print('⚠️ No configurations with <30% noise found')
    print('   Consider adjusting eps range or min_samples values')


In [ ]:
# Part 4 — Fit final DBSCAN on full data and save model+labels

# Set chosen parameters based on Part 3 results
# Option 1: Best silhouette score
best_config = df_res.sort_values('silhouette', ascending=False).iloc[0]

# Option 2: Best with low noise (uncomment if preferred)
# df_low_noise = df_res[df_res['noise_pct'] < 30].sort_values('silhouette', ascending=False)
# if len(df_low_noise) > 0:
#     best_config = df_low_noise.iloc[0]
# else:
#     print('⚠️ No low-noise config, using best silhouette')
#     best_config = df_res.sort_values('silhouette', ascending=False).iloc[0]

CHOSEN_EPS = float(best_config['eps'])
CHOSEN_MIN_SAMPLES = int(best_config['min_samples'])

print('='*70)
print('FINAL DBSCAN CONFIGURATION')
print('='*70)
print(f'eps:         {CHOSEN_EPS:.6f}')
print(f'min_samples: {CHOSEN_MIN_SAMPLES}')
print(f'metric:      euclidean' + (' (on normalized data = cosine)' if USE_COSINE_DISTANCE else ''))
print(f'Expected clusters: {int(best_config["n_clusters"])}')
print(f'Expected noise:    {best_config["noise_pct"]:.1f}%')
print(f'Expected silhouette: {best_config["silhouette"]:.4f}')
print('='*70)

print(f'\n🔄 Fitting DBSCAN on FULL dataset ({emb.shape[0]:,} samples)...')
print('   ℹ️ Using FULL data (including duplicates)')
print('   ℹ️ Duplicates will automatically cluster together (distance = 0)')
print('⏳ This may take a while... Monitor CPU usage in htop/Task Manager')
start_time = time.time()
# n_jobs=-1 to use all CPU cores
model = DBSCAN(eps=CHOSEN_EPS, min_samples=CHOSEN_MIN_SAMPLES, n_jobs=-1, metric='euclidean')
labels_full = model.fit_predict(emb)
elapsed = time.time() - start_time
print(f'✓ DBSCAN completed in {elapsed/60:.1f} minutes')

# Save labels and model
out_model = Path('dbscan_model.pkl')
joblib.dump(model, out_model)
np.save('dbscan_labels.npy', labels_full)
np.save('dbscan_config.npy', np.array([CHOSEN_EPS, CHOSEN_MIN_SAMPLES, 1 if USE_COSINE_DISTANCE else 0]))

print(f'\n✓ Saved: {out_model}')
print(f'✓ Saved: dbscan_labels.npy ({len(labels_full):,} labels)')
print(f'✓ Saved: dbscan_config.npy (eps, min_samples, use_cosine)')


In [ ]:
# Part 5 — Evaluation and cluster analysis

from collections import Counter
counts = Counter(labels_full)

print('='*70)
print('CLUSTER ANALYSIS')
print('='*70)
print('\nLabel counts (including noise -1):')
for lbl, cnt in sorted(counts.items()):
    pct = (cnt / len(labels_full)) * 100
    cluster_type = 'NOISE' if lbl == -1 else f'Cluster {lbl}'
    print(f' - {cluster_type:12s}: {cnt:8,} samples ({pct:5.2f}%)')

# Separate noise and clusters
n_noise = counts.get(-1, 0)
n_clusters = len(set(labels_full) - {-1})
n_clustered = len(labels_full) - n_noise

print(f'\nSummary:')
print(f'  Total samples:    {len(labels_full):,}')
print(f'  Clusters found:   {n_clusters}')
print(f'  Clustered points: {n_clustered:,} ({(n_clustered/len(labels_full)*100):.1f}%)')
print(f'  Noise points:     {n_noise:,} ({(n_noise/len(labels_full)*100):.1f}%)')

# Compute metrics on sample or full dataset
n = emb.shape[0]
if n > SAMPLE_FOR_METRICS:
    print(f'\n📊 Computing metrics on sample ({SAMPLE_FOR_METRICS:,} samples)...')
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.choice(n, SAMPLE_FOR_METRICS, replace=False)
    lbls_s = labels_full[idx]
    emb_s = emb[idx]
else:
    print(f'\n📊 Computing metrics on full dataset...')
    emb_s = emb
    lbls_s = labels_full

if len(set(lbls_s) - {-1}) > 1:
    try:
        sil = silhouette_score(emb_s, lbls_s)
        print(f'  Silhouette Score: {sil:.4f}')
    except Exception as e:
        print(f'  Silhouette Score: N/A ({e})')
    
    try:
        dbi = davies_bouldin_score(emb_s, lbls_s)
        print(f'  Davies-Bouldin Index: {dbi:.4f} (lower is better)')
    except Exception as e:
        print(f'  Davies-Bouldin Index: N/A ({e})')
else:
    print('⚠️ Not enough clusters (excluding noise) to compute metrics')

# Visualize cluster size distribution (excluding noise)
cluster_labels = [lbl for lbl in labels_full if lbl != -1]
if len(cluster_labels) > 0:
    cluster_counts = Counter(cluster_labels)
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.bar(cluster_counts.keys(), cluster_counts.values())
    plt.xlabel('Cluster ID')
    plt.ylabel('Number of samples')
    plt.title('Cluster Size Distribution')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    sizes = list(cluster_counts.values())
    plt.hist(sizes, bins=min(20, len(sizes)), edgecolor='black')
    plt.xlabel('Cluster size')
    plt.ylabel('Frequency')
    plt.title('Cluster Size Histogram')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'\nCluster size statistics:')
    print(f'  Min:    {min(sizes):,}')
    print(f'  Max:    {max(sizes):,}')
    print(f'  Mean:   {np.mean(sizes):,.1f}')
    print(f'  Median: {np.median(sizes):,.1f}')


## Configuration Examples & Recommended Settings

### **🆕 What's New in This Version:**

1. ✅ **Cosine Distance Support** - Normalize embeddings for better similarity (default: ON)
2. ✅ **Duplicate Handling** - Automatically detects and handles duplicate embeddings
3. ✅ **Smart k-distance** - Computes eps from unique samples when duplicates >50%
4. ✅ **Realistic min_samples** - Updated from 3-12 to 5-30 (better for log data)
5. ✅ **Better parameter range** - eps from 75th to 99th percentile (avoids too much noise)

### **Quick Start: How to Use This Notebook**

Edit **Cell 2** and configure:

```python
# Input files
INPUT_FILES = [
    Path("/your/path/to/embeddings.npy"),
]

# Key parameters
KNN_NEIGHBORS = 20              # min_samples for k-distance (realistic value)
USE_COSINE_DISTANCE = True      # Normalize for cosine distance (RECOMMENDED)
HANDLE_DUPLICATES = True        # Smart duplicate handling (RECOMMENDED)
```

### **⚙️ Configuration Options:**

**USE_COSINE_DISTANCE** (Default: `True`)
- ✅ `True`: Normalize embeddings → Euclidean = Cosine distance (BEST for BERT embeddings)
- ❌ `False`: Raw Euclidean distance (not recommended for high-dim embeddings)

**HANDLE_DUPLICATES** (Default: `True`)
- ✅ `True`: Auto-detect duplicates, compute eps from unique samples if >50% duplicates
- ❌ `False`: Use all samples (may result in k-distance = 0 if many duplicates)

**KNN_NEIGHBORS** (Default: `20`)
- Range: 10-50 for log data
- Too small (< 5): eps estimation unstable
- Too large (> 100): only captures very dense regions

---

### **Understanding Duplicate Behavior:**

**What happens if your data has many duplicates?**

```
Scenario: 90% of embeddings are duplicates
├─ Without HANDLE_DUPLICATES:
│  └─ k-distance = 0 for most points → eps unusable
│
└─ With HANDLE_DUPLICATES (RECOMMENDED):
   ├─ Step 1: Detect duplicate ratio (90%)
   ├─ Step 2: Compute k-distance from UNIQUE samples only
   ├─ Step 3: Get meaningful eps (e.g., 0.5)
   └─ Step 4: Apply DBSCAN to FULL dataset
       └─ Duplicates automatically cluster together ✅
       └─ Cluster size reflects frequency (large = frequent = normal)
       └─ Small clusters / noise = rare patterns = anomalies
```

**Why keep duplicates in final DBSCAN?**
- Duplicate frequency = important information!
- High-frequency logs → Large clusters → Normal behavior
- Low-frequency logs → Small clusters/noise → Potential anomalies

---

### **Example Configurations:**

#### **1. BGL with Duplicate Handling (RECOMMENDED)**
```python
INPUT_FILES = [
    Path("/.../after_preprocessed_bgl_embeddings.npy"),
]
KNN_NEIGHBORS = 20
USE_COSINE_DISTANCE = True
HANDLE_DUPLICATES = True
SAMPLE_FOR_KDIST = 500000
```
**Best for:** Raw 768-dim embeddings with duplicates  
**Time:** 30-45 minutes | **Quality:** ⭐⭐⭐⭐⭐

---

#### **2. BGL PCA256 (Fastest)**
```python
INPUT_FILES = [
    Path("/.../after_preprocessed_bgl_pca256_embeddings.npy"),
]
KNN_NEIGHBORS = 15
USE_COSINE_DISTANCE = True
SAMPLE_FOR_KDIST = 200000
```
**Best for:** Speed without quality loss  
**Time:** 10-15 minutes | **Quality:** ⭐⭐⭐⭐⭐

---

#### **3. Thunderbird PCA128 (Large Dataset)**
```python
INPUT_FILES = [
    Path("/.../after_preprocessed_thunderbird_pca128_embeddings.npy"),
]
KNN_NEIGHBORS = 15
USE_COSINE_DISTANCE = True
HANDLE_DUPLICATES = True
SAMPLE_FOR_KDIST = 200000
SAMPLE_FOR_METRICS = 50000
```
**Best for:** Ultra-large datasets (200M+ rows)  
**Time:** 2-4 hours | **Quality:** ⭐⭐⭐⭐

---

### **Parameter Tuning Guidelines:**

**eps (epsilon):**
- Determined from k-distance plot (75th-99th percentile)
- Smaller eps → More clusters, more noise, tighter clusters
- Larger eps → Fewer clusters, less noise, looser clusters
- If k-distance = 0: Enable `HANDLE_DUPLICATES`

**min_samples:**
- Recommended: **10-30** for log embeddings (updated from old 3-12)
- Smaller (5-10): Detect small anomaly patterns
- Medium (15-20): Balanced (RECOMMENDED)
- Larger (25-50): Only robust, frequent patterns

**Trade-offs:**
- High silhouette + high noise → Too sensitive
- Low noise + few clusters → Too loose
- **Target:** 10-30% noise with silhouette > 0.3

---

### **Troubleshooting:**

**Problem: k-distance all zeros**
```python
# Solution 1: Enable duplicate handling
HANDLE_DUPLICATES = True

# Solution 2: Enable cosine distance
USE_COSINE_DISTANCE = True

# Solution 3: Reduce KNN_NEIGHBORS
KNN_NEIGHBORS = 10  # Instead of 50
```

**Problem: 95% noise in results**
```python
# Solution: Increase eps range
# In Cell 5, the grid now uses 75th-99th percentile
# If still too much noise, manually set:
eps_min = np.percentile(k_dist_clean, 90)
eps_max = np.percentile(k_dist_clean, 99)
```

**Problem: Only 1-2 clusters**
```python
# Solution: Decrease eps or increase min_samples
min_samples_values = [10, 15, 20, 25, 30]  # Try larger values
```

---

### **Workflow:**
```
1. Run Cell 2: Configure & verify files
   └─> Check file paths and sizes
   
2. Run Cell 3: File diagnostics
   └─> Estimate runtime
   
3. Run Cell 4: Load & analyze
   ├─> Normalization (if enabled)
   ├─> Duplicate analysis
   ├─> k-distance plot from unique samples
   └─> Observe elbow point & statistics
   
4. Run Cell 5: Parameter search
   ├─> Auto grid from k-distance stats
   └─> Review top configs (silhouette + noise %)
   
5. Run Cell 6: Fit final model on FULL data
   └─> Includes duplicates (frequency preserved)
   
6. Run Cell 7: Evaluate results
   ├─> Cluster size distribution
   └─> Identify anomaly candidates
```

---

### **Output Files:**
- `dbscan_model.pkl` - Trained DBSCAN model
- `dbscan_labels.npy` - Cluster labels for FULL dataset (including -1 for noise)
- `dbscan_config.npy` - Final [eps, min_samples, use_cosine] values

### **Anomaly Detection Strategy:**
1. **Noise points (label -1)** → Primary anomaly candidates
2. **Small clusters (< 0.1% of data)** → Rare patterns, potential anomalies
3. **Large clusters** → Normal/frequent behavior

**For inference on new data:**
```python
# Load model and config
model = joblib.load('dbscan_model.pkl')
config = np.load('dbscan_config.npy')
eps, min_samples, use_cosine = config

# Prepare new embeddings
new_emb = np.load('new_embeddings.npy')
if use_cosine:
    from sklearn.preprocessing import normalize
    new_emb = normalize(new_emb, norm='l2')

# Predict (Note: DBSCAN requires fit_predict, not predict)
# For truly new data, use distance to existing cluster cores
new_labels = model.fit_predict(new_emb)
```

---

### **Key Improvements Over Previous Version:**

| Feature | Old | New |
|---------|-----|-----|
| min_samples | 3-12 (too small) | 5-30 (realistic) |
| k-distance handling | All samples | Smart: unique if >50% dup |
| Distance metric | Euclidean only | Cosine (normalized) |
| eps range | 50-95th percentile | 75-99th percentile |
| Duplicate info | Lost | Preserved in frequency |

**Result:** Better eps estimation, meaningful clusters, preserved frequency information for anomaly detection.
